# GĐ1 — TiMePReSt vs PipeDream, VGG-16 / CIFAR-100 (mô phỏng pipeline W=2 trên 1 GPU)
Chạy lần lượt từng cell. Mỗi cell chỉ gọi lệnh; mọi tham số nằm trong `configs/`.
Dán lại output của cell 3 (checks), 4 (quick) và bảng cuối cell 6 cho agent.

## 1. Drive + repo + cài đặt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO = '/content/timeprest-reproduction'
if not os.path.exists(REPO):
    !git clone https://github.com/cotda/timeprest-reproduction.git {REPO}
%cd {REPO}
!git pull --ff-only
!git log --oneline -1
!pip install -q -e ".[test]"

## 2. Môi trường

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python -c "import torch, torchvision; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| torchvision', torchvision.__version__, '| gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"
!ls -la /content/drive/MyDrive/datasets/cifar-100-python  # phải có train, test, meta


## 3. Unit test (CPU) + 7 check trước khi chạy dài (~5–10 phút)

In [ ]:
!python -m pytest -q
!python -m timeprest.checks --all --config configs/quick.yaml

## 4. Chạy ngắn (subset 5000 ảnh, 2 epoch)

In [ ]:
!rm -rf results/runs/quick_timeprest
!python -m timeprest.train --config configs/quick.yaml

## 5. Chạy dài (160 epoch mỗi hệ, checkpoint trên Drive)
Nếu Colab ngắt: chạy lại cell 1, rồi chạy lại đúng cell này (đã có `--resume`).

In [ ]:
!python -m timeprest.train --config configs/full_timeprest.yaml --resume

In [ ]:
!python -m timeprest.train --config configs/full_pipedream.yaml --resume

## 6. So sánh (giống Fig.4 của paper) — dán bảng cho agent

In [ ]:
RUNS = '/content/drive/MyDrive/timeprest/runs'
!python -m timeprest.plot --runs {RUNS}/cifar100_vgg16_timeprest {RUNS}/cifar100_vgg16_pipedream --out {RUNS}/compare_cifar100
from IPython.display import Image
Image(f'{RUNS}/compare_cifar100.png')